# True Value Promotion Feature – V1 Development Notebook

This notebook documents the development of the True Value Promotion feature for DiscountMate. The goal of this feature is to identify promotions that offer meaningful customer value, rather than simply relying on retailer “special” labels.

The notebook follows the full workflow from retailer data loading and harmonisation, through feature engineering and category preparation, to category-aware scoring and customer-facing discount classification.

In [ ]:
import pandas as pd
import numpy as np
import re

## 1 Data Loading and Initial Inspection

In this section, the retailer datasets are imported into Python and checked at a high level. The purpose of this step is to understand the structure of each source file, including row counts, column names, and data types, before starting any harmonisation work.

In [2]:
# Load the Coles dataset
coles_df = pd.read_csv(r"C:\Shazza\T1 2026\Capstone Project B\#_True_Value_Promotion\Datasets_23_Mar\Coles.csv")

# Load the Woolworths dataset
woolworths_df = pd.read_csv(r"C:\Shazza\T1 2026\Capstone Project B\#_True_Value_Promotion\Datasets_23_Mar\Woolworths.csv")

# Load the IGA dataset
iga_df = pd.read_csv(r"C:\Shazza\T1 2026\Capstone Project B\#_True_Value_Promotion\Datasets_23_Mar\IGA.csv")

# Quick preview
print("Coles:")
print(coles_df.head())

print("\nWoolworths:")
print(woolworths_df.head())

print("\nIGA:")
print(iga_df.head())

Coles:
  Brand_Searched  ProductId                  Name  Brand  \
0          Coles    5191256          Strawberries  Coles   
1          Coles    8150288       Full Cream Milk  Coles   
2          Coles    9006560               Carrots  Coles   
3          Coles    4584071       Iceberg Lettuce  Coles   
4          Coles    9161519  2-ply Facial Tissues  Coles   

                           Description      Size  Price_Now  Price_Was  \
0      STRAWBERRIES:BERRIES:.:250 GRAM      250g       4.50        0.0   
1             COLES FULL CREAM MILK 3L        3L       4.65        0.0   
2               CARROTS:PREPACK:.:1 KG       1Kg       1.70        0.0   
3            LETTUCE ICEBERG::.:1 EACH    1 Each       2.90        0.0   
4  COLES 2-PLY FACIAL TISSUES 224 PACK  224 Pack       1.90        0.0   

   SaveAmount SaveStatement  ...  TradeProfitCentre        CategoryGroup  \
0         0.0           NaN  ...         FRESH PROD                FRUIT   
1         0.0           NaN  ...   

In [3]:
print("Coles shape:", coles_df.shape)
print("Woolworths shape:", woolworths_df.shape)
print("IGA shape:", iga_df.shape)

print("\nColes columns:")
print(coles_df.columns)

print("\nWoolworths columns:")
print(woolworths_df.columns)

print("\nIGA columns:")
print(iga_df.columns)

Coles shape: (17075, 37)
Woolworths shape: (30469, 53)
IGA shape: (6011, 65)

Coles columns:
Index(['Brand_Searched', 'ProductId', 'Name', 'Brand', 'Description', 'Size',
       'Price_Now', 'Price_Was', 'SaveAmount', 'SaveStatement', 'UnitPrice',
       'UnitQuantity', 'UnitMeasure', 'Comparable', 'PromotionType',
       'SpecialType', 'OnlineSpecial', 'OfferDescription',
       'MultiBuy_MinQuantity', 'MultiBuy_Reward', 'Availability',
       'AvailabilityType', 'AvailableQuantity', 'RetailLimit',
       'PromotionalLimit', 'LiquorRestricted', 'TobaccoRestricted',
       'TradeProfitCentre', 'CategoryGroup', 'Category', 'SubCategory',
       'ClassName', 'OnlineAisle', 'OnlineCategory', 'OnlineSubCategory',
       'ImageUri', 'Timestamp'],
      dtype='object')

Woolworths columns:
Index(['Brand_Searched', 'Stockcode', 'Barcode', 'Name', 'DisplayName',
       'UrlFriendlyName', 'Description', 'FullDescription', 'PackageSize',
       'Price', 'InstorePrice', 'WasPrice', 'InstoreWasPri

In [4]:
print("\nColes info:")
print(coles_df.info())

print("\nWoolworths info:")
print(woolworths_df.info())

print("\nIGA info:")
print(iga_df.info())


Coles info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17075 entries, 0 to 17074
Data columns (total 37 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Brand_Searched        17075 non-null  object 
 1   ProductId             17075 non-null  int64  
 2   Name                  17075 non-null  object 
 3   Brand                 17075 non-null  object 
 4   Description           17075 non-null  object 
 5   Size                  17074 non-null  object 
 6   Price_Now             17075 non-null  float64
 7   Price_Was             17075 non-null  float64
 8   SaveAmount            17075 non-null  float64
 9   SaveStatement         4361 non-null   object 
 10  UnitPrice             17075 non-null  float64
 11  UnitQuantity          17075 non-null  int64  
 12  UnitMeasure           16297 non-null  object 
 13  Comparable            16558 non-null  object 
 14  PromotionType         9607 non-null   object 
 15  Specia

## 2. Retailer-Specific Column Selection

Each retailer dataset contains many fields that are not directly required for the feature. At this stage, only the most relevant columns are retained, focusing on product identity, pricing, promotion signals, category information, and timestamps.

The aim is to reduce noise and prepare each dataset for schema alignment.

### 2.1. Coles columns to keep

In [7]:
coles_keep = [
    'ProductId', 'Name', 'Brand', 'Size',
    'Price_Now', 'Price_Was', 'SaveAmount',
    'Comparable', 'PromotionType', 'OnlineSpecial',
    'CategoryGroup', 'Category', 'SubCategory',
    'ImageUri', 'Timestamp'
]

coles_clean = coles_df[coles_keep].copy()
print(coles_clean.head())
print(coles_clean.shape)

   ProductId                  Name  Brand      Size  Price_Now  Price_Was  \
0    5191256          Strawberries  Coles      250g       4.50        0.0   
1    8150288       Full Cream Milk  Coles        3L       4.65        0.0   
2    9006560               Carrots  Coles       1Kg       1.70        0.0   
3    4584071       Iceberg Lettuce  Coles    1 Each       2.90        0.0   
4    9161519  2-ply Facial Tissues  Coles  224 Pack       1.90        0.0   

   SaveAmount    Comparable PromotionType  OnlineSpecial        CategoryGroup  \
0         0.0   $18.00/ 1kg       SPECIAL          False                FRUIT   
1         0.0     $1.55/ 1L      EVERYDAY          False                DAIRY   
2         0.0    $1.70/ 1kg       SPECIAL          False     VEGETABLES/SALAD   
3         0.0    $2.90/ 1ea       SPECIAL          False     VEGETABLES/SALAD   
4         0.0  $0.85/ 100ea      EVERYDAY          False  HEALTH  BEAUTY BABY   

          Category         SubCategory        Imag

### 2.2. Woolworths columns to keep

In [9]:
woolworths_keep = [
    'Stockcode', 'Name', 'DisplayName', 'PackageSize',
    'Price', 'WasPrice', 'SavingsAmount',
    'CupString', 'IsOnSpecial', 'IsHalfPrice',
    'SapDepartmentName', 'SapCategoryName', 'SapSubCategoryName',
    'PromotionType', 'LargeImageFile', 'Timestamp'
]

woolworths_clean = woolworths_df[woolworths_keep].copy()
print(woolworths_clean.head())
print(woolworths_clean.shape)

    Stockcode                                          Name  \
0  1114830817    LEGO 71769 - Ninjago Cole's Dragon Cruiser   
1  1127190213      LEGO NINJAGO 71769 Cole's Dragon Cruiser   
2  1118141455            Tyler Johnson Was Here - Jay Coles   
3  1126584955                 Esselte R/Band Col Sz14 100gm   
4  1115990293  LEGO 71782 - Ninjago Cole's Earth Dragon EVO   

                                    DisplayName PackageSize   Price  WasPrice  \
0    LEGO 71769 - Ninjago Cole's Dragon Cruiser         1EA  249.99    249.99   
1      LEGO NINJAGO 71769 Cole's Dragon Cruiser         NaN  296.26    296.26   
2            Tyler Johnson Was Here - Jay Coles         NaN   42.25     42.25   
3                 Esselte R/Band Col Sz14 100gm         NaN   29.00     29.00   
4  LEGO 71782 - Ninjago Cole's Earth Dragon EVO         1EA   79.99     79.99   

   SavingsAmount CupString  IsOnSpecial  IsHalfPrice SapDepartmentName  \
0            0.0       NaN        False        False        

### 2.3. IGA columns to keep

In [11]:
iga_keep = [
    'iga_product_id', 'name', 'brand_name',
    'price_numeric', 'was_price_numeric',
    'price_per_unit', 'price_source',
    'iga_default_category', 'iga_categories',
    'primary_image_url', 'scraped_at'
]

iga_clean = iga_df[iga_keep].copy()
print(iga_clean.head())
print(iga_clean.shape)

   iga_product_id                                name    brand_name  \
0        764109.0        BabyBoo Unscented Baby Wipes       BabyBoo   
1        764125.0  BabyBoo Baby Wipes Lightly Scented       BabyBoo   
2        931133.0        IGA Large Black Reusable Bag           IGA   
3        647763.0             Community Co Serviettes  Community Co   
4        851142.0                 Community Co Walnut  Community Co   

   price_numeric  was_price_numeric price_per_unit price_source  \
0           2.20                NaN     $0.03 each      regular   
1           2.20                NaN     $0.03 each      regular   
2           1.25                NaN            NaN      regular   
3           2.50                NaN     $0.10 each      regular   
4           4.15                NaN     $3.19/100g      regular   

                                iga_default_category  \
0  [{"category": "Baby Wipes and Changing", "cate...   
1  [{"category": "Baby Wipes and Changing", "cate...   
2 

## 3. Schema Harmonisation Across Retailers

Because Coles, Woolworths, and IGA use different column names and structures, the selected fields are renamed into a shared schema. This allows the retailer datasets to be combined into a single master dataset for downstream analysis.

At this point, the focus is not yet on scoring promotions, but on making the underlying data consistent and comparable.

In [14]:
coles_clean = coles_clean.rename(columns={
    'ProductId': 'product_id',
    'Name': 'product_name',
    'Brand': 'brand',
    'Size': 'size',
    'Price_Now': 'current_price',
    'Price_Was': 'original_price',
    'SaveAmount': 'saving_amount',
    'Comparable': 'unit_price_text',
    'PromotionType': 'promotion_type',
    'OnlineSpecial': 'promo_flag',
    'CategoryGroup': 'category_group',
    'Category': 'category',
    'SubCategory': 'subcategory',
    'ImageUri': 'image_url',
    'Timestamp': 'scrape_timestamp'
})

coles_clean['retailer'] = 'Coles'

### Woolworths rename

In [16]:
woolworths_clean = woolworths_clean.rename(columns={
    'Stockcode': 'product_id',
    'DisplayName': 'product_name',
    'PackageSize': 'size',
    'Price': 'current_price',
    'WasPrice': 'original_price',
    'SavingsAmount': 'saving_amount',
    'CupString': 'unit_price_text',
    'IsOnSpecial': 'promo_flag',
    'PromotionType': 'promotion_type',
    'SapDepartmentName': 'category_group',
    'SapCategoryName': 'category',
    'SapSubCategoryName': 'subcategory',
    'LargeImageFile': 'image_url',
    'Timestamp': 'scrape_timestamp'
})

woolworths_clean['retailer'] = 'Woolworths'

### IGA rename

In [18]:
iga_clean = iga_clean.rename(columns={
    'iga_product_id': 'product_id',
    'name': 'product_name',
    'brand_name': 'brand',
    'price_numeric': 'current_price',
    'was_price_numeric': 'original_price',
    'price_per_unit': 'unit_price_text',
    'price_source': 'promotion_type',
    'iga_default_category': 'category',
    'iga_categories': 'subcategory',
    'primary_image_url': 'image_url',
    'scraped_at': 'scrape_timestamp'
})

iga_clean['retailer'] = 'IGA'

##### Because IGA does not have some exact columns, add them as blanks:

In [21]:
iga_clean['size'] = np.nan
iga_clean['saving_amount'] = np.nan
iga_clean['promo_flag'] = np.nan
iga_clean['category_group'] = np.nan

##### For Woolworths, keep one product name column only:

In [23]:
woolworths_clean = woolworths_clean.drop(columns=['Name'])

## 4. Standardising Dataset Structure

Not every retailer provides the same fields. To ensure all datasets can be merged cleanly, missing columns are added where needed and the final column order is standardised.

This step creates a consistent structure across all retailer sources.

In [25]:

woolworths_clean['brand'] = np.nan

final_cols = [
    'retailer', 'product_id', 'product_name', 'brand', 'size',
    'current_price', 'original_price', 'saving_amount',
    'unit_price_text', 'promo_flag', 'promotion_type',
    'category_group', 'category', 'subcategory',
    'image_url', 'scrape_timestamp'
]

coles_clean = coles_clean[final_cols]
woolworths_clean = woolworths_clean[final_cols]
iga_clean = iga_clean[final_cols]

In [27]:
print(coles_clean.columns)
print(woolworths_clean.columns)
print(iga_clean.columns)

Index(['retailer', 'product_id', 'product_name', 'brand', 'size',
       'current_price', 'original_price', 'saving_amount', 'unit_price_text',
       'promo_flag', 'promotion_type', 'category_group', 'category',
       'subcategory', 'image_url', 'scrape_timestamp'],
      dtype='object')
Index(['retailer', 'product_id', 'product_name', 'brand', 'size',
       'current_price', 'original_price', 'saving_amount', 'unit_price_text',
       'promo_flag', 'promotion_type', 'category_group', 'category',
       'subcategory', 'image_url', 'scrape_timestamp'],
      dtype='object')
Index(['retailer', 'product_id', 'product_name', 'brand', 'size',
       'current_price', 'original_price', 'saving_amount', 'unit_price_text',
       'promo_flag', 'promotion_type', 'category_group', 'category',
       'subcategory', 'image_url', 'scrape_timestamp'],
      dtype='object')


## 5. Master Dataset Creation

Once the three retailer datasets share the same structure, they are combined into a unified master dataset. This master table becomes the main working dataset for the feature.

The output of this step confirms that the merge has worked and helps identify missing values that need to be addressed in later cleaning steps.

In [29]:
# Combine all three retailer datasets into one master dataframe
master_df = pd.concat([coles_clean, woolworths_clean, iga_clean], ignore_index=True)

# Quick check
print("Master dataset shape:", master_df.shape)
print(master_df.head())

Master dataset shape: (53555, 16)
  retailer  product_id          product_name  brand      size  current_price  \
0    Coles   5191256.0          Strawberries  Coles      250g           4.50   
1    Coles   8150288.0       Full Cream Milk  Coles        3L           4.65   
2    Coles   9006560.0               Carrots  Coles       1Kg           1.70   
3    Coles   4584071.0       Iceberg Lettuce  Coles    1 Each           2.90   
4    Coles   9161519.0  2-ply Facial Tissues  Coles  224 Pack           1.90   

   original_price  saving_amount unit_price_text promo_flag promotion_type  \
0             0.0            0.0     $18.00/ 1kg        0.0        SPECIAL   
1             0.0            0.0       $1.55/ 1L        0.0       EVERYDAY   
2             0.0            0.0      $1.70/ 1kg        0.0        SPECIAL   
3             0.0            0.0      $2.90/ 1ea        0.0        SPECIAL   
4             0.0            0.0    $0.85/ 100ea        0.0       EVERYDAY   

        category

In [31]:
print(master_df.info())
print(master_df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53555 entries, 0 to 53554
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   retailer          53555 non-null  object 
 1   product_id        53555 non-null  float64
 2   product_name      53555 non-null  object 
 3   brand             22987 non-null  object 
 4   size              42865 non-null  object 
 5   current_price     53054 non-null  float64
 6   original_price    49159 non-null  float64
 7   saving_amount     47043 non-null  float64
 8   unit_price_text   39898 non-null  object 
 9   promo_flag        47544 non-null  object 
 10  promotion_type    46087 non-null  object 
 11  category_group    30822 non-null  object 
 12  category          36833 non-null  object 
 13  subcategory       36833 non-null  object 
 14  image_url         53389 non-null  object 
 15  scrape_timestamp  53555 non-null  object 
dtypes: float64(4), object(12)
memory usage: 

## 6. Data Cleaning and Feature Derivation

This section prepares the master dataset for scoring by cleaning text and numeric fields, standardising timestamps, and deriving feature-ready attributes.

Key outputs from this stage include:
- discount percentage
- measurable savings
- promotion indicators

This stage is important because promotion labels alone are not enough; the feature needs measurable pricing signals to assess true value.


At this stage, an important decision was made to distinguish between products that were simply marked as promoted by the retailer and products with a measurable numeric saving. This helped create a more reliable base for the feature.

In [33]:
# Convert scrape timestamp to datetime
master_df['scrape_timestamp'] = pd.to_datetime(master_df['scrape_timestamp'], errors='coerce')

# Clean text fields
master_df['product_name'] = master_df['product_name'].astype(str).str.strip()
master_df['brand'] = master_df['brand'].astype(str).str.strip()
master_df['size'] = master_df['size'].astype(str).str.strip()
master_df['unit_price_text'] = master_df['unit_price_text'].astype(str).str.strip()
master_df['promotion_type'] = master_df['promotion_type'].astype(str).str.strip()
master_df['category_group'] = master_df['category_group'].astype(str).str.strip()
master_df['category'] = master_df['category'].astype(str).str.strip()
master_df['subcategory'] = master_df['subcategory'].astype(str).str.strip()

# Convert price-related columns to numeric
master_df['current_price'] = pd.to_numeric(master_df['current_price'], errors='coerce')
master_df['original_price'] = pd.to_numeric(master_df['original_price'], errors='coerce')
master_df['saving_amount'] = pd.to_numeric(master_df['saving_amount'], errors='coerce')

# Derive saving_amount where possible
master_df['saving_amount'] = np.where(
    master_df['saving_amount'].isna() &
    master_df['original_price'].notna() &
    master_df['current_price'].notna(),
    master_df['original_price'] - master_df['current_price'],
    master_df['saving_amount']
)

# If original price is 0, treat it as missing
master_df['original_price'] = master_df['original_price'].replace(0, np.nan)

# Recalculate saving_amount again after fixing original_price
master_df['saving_amount'] = np.where(
    master_df['original_price'].notna() &
    master_df['current_price'].notna(),
    master_df['original_price'] - master_df['current_price'],
    master_df['saving_amount']
)

# Replace negative savings with 0
master_df['saving_amount'] = master_df['saving_amount'].apply(
    lambda x: x if pd.isna(x) or x >= 0 else 0
)

# Create discount percentage
master_df['discount_percent'] = np.where(
    master_df['original_price'].notna() & (master_df['original_price'] > 0),
    (master_df['saving_amount'] / master_df['original_price']) * 100,
    0
)

master_df['discount_percent'] = master_df['discount_percent'].round(2)

# Standardised promo flag
master_df['promo_flag_final'] = np.where(
    (master_df['saving_amount'] > 0) |
    (master_df['promo_flag'] == True) |
    (master_df['promotion_type'].str.lower().isin(['tpr', 'special', 'down down', 'multibuy'])),
    1,
    0
)

# Quick check
print(master_df[['retailer', 'product_name', 'current_price', 'original_price',
                 'saving_amount', 'discount_percent', 'promo_flag_final']].head(10))

  retailer                                product_name  current_price  \
0    Coles                                Strawberries           4.50   
1    Coles                             Full Cream Milk           4.65   
2    Coles                                     Carrots           1.70   
3    Coles                             Iceberg Lettuce           2.90   
4    Coles                        2-ply Facial Tissues           1.90   
5    Coles  No Added Hormone Beef 3 Star Regular Mince          14.00   
6    Coles                     Free Range Eggs 12 Pack           6.20   
7    Coles                      Garlic Bread Twin Pack           2.10   
8    Coles         Ultimate Cookies 40% Chocolate Chip           6.00   
9    Coles                        Squeezy Tomato Sauce           2.00   

   original_price  saving_amount  discount_percent  promo_flag_final  
0             NaN            0.0               0.0                 1  
1             NaN            0.0               0.0    

In [34]:
print(master_df[['current_price', 'original_price', 'saving_amount', 'discount_percent']].describe())

print(master_df['promo_flag_final'].value_counts(dropna=False))

print(master_df.isnull().sum())

       current_price  original_price  saving_amount  discount_percent
count   53054.000000    37098.000000   48658.000000      53054.000000
mean       46.269535       62.549418       1.068534          5.996078
std       253.273051      301.819079       5.918144         13.594038
min         0.000000        0.140000       0.000000          0.000000
25%         4.900000        6.050000       0.000000          0.000000
50%        10.000000       16.000000       0.000000          0.000000
75%        29.950000       40.000000       0.000000          0.000000
max     18329.000000    18329.000000     338.000000         80.000000
promo_flag_final
0    41254
1    12301
Name: count, dtype: int64
retailer                0
product_id              0
product_name            0
brand                   0
size                    0
current_price         501
original_price      16457
saving_amount        4897
unit_price_text         0
promo_flag           6011
promotion_type          0
category_group     

In [36]:
# Replace string 'nan' and empty strings back to real missing values
text_cols = ['brand', 'size', 'unit_price_text', 'promotion_type',
             'category_group', 'category', 'subcategory']

for col in text_cols:
    master_df[col] = master_df[col].replace(['nan', 'None', ''], np.nan)

# Check missing values again
print(master_df[text_cols].isnull().sum())

brand              30568
size               10690
unit_price_text    13657
promotion_type      7468
category_group     22733
category           16722
subcategory        16722
dtype: int64


Then create a second promo indicator

In [38]:
# Promotion label flag: product is marked/promoted in some way
master_df['is_promoted'] = np.where(
    (master_df['promo_flag'] == True) |
    (master_df['promotion_type'].astype(str).str.lower().isin(['tpr', 'special', 'down down', 'multibuy'])),
    1,
    0
)

# Measurable saving flag: product has an actual price saving
master_df['has_measurable_saving'] = np.where(
    master_df['saving_amount'] > 0,
    1,
    0
)

print(master_df[['is_promoted', 'has_measurable_saving']].sum())

is_promoted              11649
has_measurable_saving    10771
dtype: int64


## 7. Category Review Across Retailers

A meaningful promotion can vary by product category, so category preparation was treated as an important step before scoring. This section reviews the raw category structures used by each retailer in order to understand how they differ and what level of mapping will be required.

The goal here is to prepare for a shared category framework rather than use raw retailer categories directly.

#### Coles

In [41]:
print("Coles category_group:")
print(master_df[master_df['retailer'] == 'Coles']['category_group'].value_counts(dropna=False).head(20))

print("\nColes category:")
print(master_df[master_df['retailer'] == 'Coles']['category'].value_counts(dropna=False).head(20))

Coles category_group:
category_group
HEALTH  BEAUTY BABY     4587
MEAL SOLUTIONS          3784
IMPULSE                 2643
DAIRY                   2211
HOME CARE               1918
GENERAL MERCHANDISE      580
LIQUOR                   319
BAKERY CAKES/YEAST       316
FRUIT                    150
VEGETABLES/SALAD         141
IN-STORE BH/BOUTIQUE     118
SMALLGOODS/POULTRY        74
BASIC APPAREL             68
VALUE ADDED               65
POULTRY                   37
BEEF/VEAL                 19
PORK/HAMS/BACON           13
LAMB                      10
SEAFOOD                    8
APPAREL OUTERWEAR          7
Name: count, dtype: int64

Coles category:
category
HAIR CARE               880
BISCUITS & COOKIES      467
SKIN CARE               447
VITAMINS                447
COSMETICS/TOILETRIES    409
HEALTH FOODS            380
BAKING MIXES            369
CEREAL                  330
CHILLED DESSERTS        327
MEDICINAL PRODUCTS      326
COFFEE                  319
CONFECTIONERY          

#### Woolworths

In [43]:
print("Woolworths category_group:")
print(master_df[master_df['retailer'] == 'Woolworths']['category_group'].value_counts(dropna=False).head(20))

print("\nWoolworths category:")
print(master_df[master_df['retailer'] == 'Woolworths']['category'].value_counts(dropna=False).head(20))

Woolworths category_group:
category_group
NaN                    16722
GROCERIES              11182
FRESH CONVENIENCE       1248
GENERAL MERCHANDISE      729
PROPRIETARY BAKERY       276
BAKEHOUSE                133
FRUIT AND VEG             91
FRONT OF STORE            46
DELI SERVICE              38
NON TRADING                2
SEAFOOD SERVICE            2
Name: count, dtype: int64

Woolworths category:
category
NaN                       16722
TOILETRIES                 2179
HEALTH CARE                 900
CLEANSING                   675
COOKING NEEDS               658
BISCUITS                    596
PET FOOD                    554
ETHNIC / GOURMET FOOD       530
CONFECTIONERY               452
BABY NEEDS                  425
BEVERAGES                   412
PREPARED FOODS              366
HEALTH FOODS                360
DAIRY - YOGHURT             309
PASTA / RICE                305
CONDIMENTS                  298
PROPRIETARY BAKERY          276
FROZEN MEALS                266
CARBON

#### IGA

In [45]:
print("IGA category:")
print(master_df[master_df['retailer'] == 'IGA']['category'].value_counts(dropna=False).head(20))

print("\nIGA subcategory:")
print(master_df[master_df['retailer'] == 'IGA']['subcategory'].value_counts(dropna=False).head(20))

IGA category:
category
[{"category": "Australian Beer", "categoryBreadcrumb": "Grocery/Liquor/Beer/Australian Beer", "categoryId": "29a1aee6-5cd4-4c53-a511-262426174536", "retailerId": "Australian_Beer_Food"}]                                     167
[{"category": "Flavoured Soft Drinks", "categoryBreadcrumb": "Grocery/Drinks/Soft Drinks/Flavoured Soft Drinks", "categoryId": "d6a551cf-2232-4c40-bbfd-7405b46421b6", "retailerId": "Flavoured_Soft_Drinks"}]                 120
[{"category": "Chocolate Blocks", "categoryBreadcrumb": "Grocery/Pantry/Confectionary/Chocolate Blocks", "categoryId": "d5587b61-e270-4254-82df-7fbe18135ae4", "retailerId": "Chocolate_Blocks"}]                              104
[{"category": "Premix Drinks", "categoryBreadcrumb": "Grocery/Liquor/Premix Drinks", "categoryId": "2749047c-b45a-442f-8fa9-bae6a9b3f98c", "retailerId": "Premix_Drinks_Food"}]                                                103
[{"category": "Biscuits and Cookies", "categoryBreadcrumb": "Grocery/

## 8. Creating a Shared Category Source

To support later mapping, a retailer-aware category source column is created. For Coles and Woolworths, the most suitable category field is selected. For IGA, category information is extracted from JSON-like text into a more usable format.

This step creates a cleaner starting point for shared category mapping.

In [47]:
# Create a retailer-specific raw category source
master_df['raw_main_cat'] = np.nan

# Coles: use category_group
master_df.loc[master_df['retailer'] == 'Coles', 'raw_main_cat'] = master_df['category_group']

# Woolworths: prefer category, then fallback to category_group
master_df.loc[master_df['retailer'] == 'Woolworths', 'raw_main_cat'] = (
    master_df.loc[master_df['retailer'] == 'Woolworths', 'category']
    .fillna(master_df.loc[master_df['retailer'] == 'Woolworths', 'category_group'])
)

# IGA: temporarily use category as-is for now
master_df.loc[master_df['retailer'] == 'IGA', 'raw_main_cat'] = master_df['category']

# Quick check
print(master_df[['retailer', 'raw_main_cat']].head(10))
print(master_df.groupby('retailer')['raw_main_cat'].nunique(dropna=True))

  retailer         raw_main_cat
0    Coles                FRUIT
1    Coles                DAIRY
2    Coles     VEGETABLES/SALAD
3    Coles     VEGETABLES/SALAD
4    Coles  HEALTH  BEAUTY BABY
5    Coles            BEEF/VEAL
6    Coles       MEAL SOLUTIONS
7    Coles                DAIRY
8    Coles              IMPULSE
9    Coles       MEAL SOLUTIONS
retailer
Coles          22
IGA           534
Woolworths     69
Name: raw_main_cat, dtype: int64


C:\Users\mrsha\AppData\Local\Temp\ipykernel_16524\1873712977.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['FRUIT' 'DAIRY' 'VEGETABLES/SALAD' ... 'MEAL SOLUTIONS' 'MEAL SOLUTIONS'
 'MEAL SOLUTIONS']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  master_df.loc[master_df['retailer'] == 'Coles', 'raw_main_cat'] = master_df['category_group']


In [48]:
# Create a retailer-specific raw category source
master_df['raw_main_cat'] = np.nan

# Coles: use category_group
master_df.loc[master_df['retailer'] == 'Coles', 'raw_main_cat'] = master_df['category_group']

# Woolworths: prefer category, then fallback to category_group
master_df.loc[master_df['retailer'] == 'Woolworths', 'raw_main_cat'] = (
    master_df.loc[master_df['retailer'] == 'Woolworths', 'category']
    .fillna(master_df.loc[master_df['retailer'] == 'Woolworths', 'category_group'])
)

# IGA: temporarily use category as-is for now
master_df.loc[master_df['retailer'] == 'IGA', 'raw_main_cat'] = master_df['category']

# Quick check
print(master_df[['retailer', 'raw_main_cat']].head(10))
print(master_df.groupby('retailer')['raw_main_cat'].nunique(dropna=True))

C:\Users\mrsha\AppData\Local\Temp\ipykernel_16524\1873712977.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['FRUIT' 'DAIRY' 'VEGETABLES/SALAD' ... 'MEAL SOLUTIONS' 'MEAL SOLUTIONS'
 'MEAL SOLUTIONS']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  master_df.loc[master_df['retailer'] == 'Coles', 'raw_main_cat'] = master_df['category_group']


  retailer         raw_main_cat
0    Coles                FRUIT
1    Coles                DAIRY
2    Coles     VEGETABLES/SALAD
3    Coles     VEGETABLES/SALAD
4    Coles  HEALTH  BEAUTY BABY
5    Coles            BEEF/VEAL
6    Coles       MEAL SOLUTIONS
7    Coles                DAIRY
8    Coles              IMPULSE
9    Coles       MEAL SOLUTIONS
retailer
Coles          22
IGA           534
Woolworths     69
Name: raw_main_cat, dtype: int64


#### Clean IGA’s raw category text into something usable

In [50]:
# For IGA, extract the most specific category from the JSON-like text
master_df.loc[master_df['retailer'] == 'IGA', 'raw_main_cat'] = (
    master_df.loc[master_df['retailer'] == 'IGA', 'raw_main_cat']
    .astype(str)
    .str.findall(r'"category":\s*"([^"]+)"')
    .str[-1]
)

# Quick check
print(master_df[master_df['retailer'] == 'IGA']['raw_main_cat'].value_counts(dropna=False).head(20))

raw_main_cat
Australian Beer              167
Flavoured Soft Drinks        120
Chocolate Blocks             110
Premix Drinks                103
Crisps and Crackers          101
Biscuits and Cookies         100
Chocolate Bars                95
Potato Chips                  95
Juices                        89
Sweets and Lollies            84
Multipacks                    83
Chocolate Share Packs         79
Shiraz                        78
Craft Beer                    76
Whisky                        72
Energy Drinks                 69
Ready to Eat Meals            68
Nuts and Jerky                62
Gums, Mints and Medicated     58
Packaged Bread                56
Name: count, dtype: int64


In [51]:
print("Coles raw categories:")
print(master_df[master_df['retailer'] == 'Coles']['raw_main_cat'].value_counts(dropna=False).head(20))

print("\nWoolworths raw categories:")
print(master_df[master_df['retailer'] == 'Woolworths']['raw_main_cat'].value_counts(dropna=False).head(20))

print("\nIGA raw categories:")
print(master_df[master_df['retailer'] == 'IGA']['raw_main_cat'].value_counts(dropna=False).head(20))

Coles raw categories:
raw_main_cat
HEALTH  BEAUTY BABY     4587
MEAL SOLUTIONS          3784
IMPULSE                 2643
DAIRY                   2211
HOME CARE               1918
GENERAL MERCHANDISE      580
LIQUOR                   319
BAKERY CAKES/YEAST       316
FRUIT                    150
VEGETABLES/SALAD         141
IN-STORE BH/BOUTIQUE     118
SMALLGOODS/POULTRY        74
BASIC APPAREL             68
VALUE ADDED               65
POULTRY                   37
BEEF/VEAL                 19
PORK/HAMS/BACON           13
LAMB                      10
SEAFOOD                    8
APPAREL OUTERWEAR          7
Name: count, dtype: int64

Woolworths raw categories:
raw_main_cat
NaN                       16722
TOILETRIES                 2179
HEALTH CARE                 900
CLEANSING                   675
COOKING NEEDS               658
BISCUITS                    596
PET FOOD                    554
ETHNIC / GOURMET FOOD       530
CONFECTIONERY               452
BABY NEEDS                  42

## 9. First-Pass Category Harmonisation

The cleaned retailer category values are mapped into a shared business-friendly category structure. This includes groups such as Fresh Food, Pantry, Dairy & Refrigerated, Snacks & Confectionery, Beverages, Cleaning & Household, Personal Care & Health, and others.

The purpose of this stage is to support category-aware promotion analysis, since discount behaviour varies across product types.


This stage required iteration because some retailer categories were broad, missing, or encoded differently. Rather than aiming for a perfect taxonomy immediately, the focus was on creating a practical V1 category structure that would be strong enough to support scoring.

In [53]:
# Create default category column
master_df['main_category'] = 'Other'

# Dictionary-based category mapping
category_map = {
    'FRUIT': 'Fresh Food',
    'VEGETABLES/SALAD': 'Fresh Food',
    'VALUE ADDED': 'Fresh Food',

    'DAIRY': 'Dairy & Refrigerated',
    'DAIRY - YOGHURT': 'Dairy & Refrigerated',
    'Ready to Eat Meals': 'Dairy & Refrigerated',
    'DELI CONVENIENCE': 'Dairy & Refrigerated',

    'MEAL SOLUTIONS': 'Pantry',
    'COOKING NEEDS': 'Pantry',
    'ETHNIC / GOURMET FOOD': 'Pantry',
    'HEALTH FOODS': 'Pantry',
    'PASTA / RICE': 'Pantry',
    'CONDIMENTS': 'Pantry',
    'PREPARED FOODS': 'Pantry',

    'IMPULSE': 'Snacks & Confectionery',
    'BISCUITS': 'Snacks & Confectionery',
    'CONFECTIONERY': 'Snacks & Confectionery',
    'Biscuits and Cookies': 'Snacks & Confectionery',
    'Crisps and Crackers': 'Snacks & Confectionery',
    'Chocolate Blocks': 'Snacks & Confectionery',
    'Chocolate Bars': 'Snacks & Confectionery',
    'Potato Chips': 'Snacks & Confectionery',
    'Sweets and Lollies': 'Snacks & Confectionery',
    'Chocolate Share Packs': 'Snacks & Confectionery',
    'Nuts and Jerky': 'Snacks & Confectionery',
    'Gums, Mints and Medicated': 'Snacks & Confectionery',

    'BEVERAGES': 'Beverages',
    'CARBONATED SOFT DRINKS': 'Beverages',
    'Flavoured Soft Drinks': 'Beverages',
    'Juices': 'Beverages',
    'Energy Drinks': 'Beverages',

    'FROZEN MEALS': 'Frozen',
    'Multipacks': 'Frozen',

    'BAKERY CAKES/YEAST': 'Bakery',
    'PROPRIETARY BAKERY': 'Bakery',
    'Packaged Bread': 'Bakery',

    'SMALLGOODS/POULTRY': 'Meat & Seafood',
    'POULTRY': 'Meat & Seafood',
    'BEEF/VEAL': 'Meat & Seafood',
    'PORK/HAMS/BACON': 'Meat & Seafood',
    'LAMB': 'Meat & Seafood',
    'SEAFOOD': 'Meat & Seafood',

    'HOME CARE': 'Cleaning & Household',
    'CLEANSING': 'Cleaning & Household',

    'HEALTH  BEAUTY BABY': 'Personal Care & Health',
    'TOILETRIES': 'Personal Care & Health',
    'HEALTH CARE': 'Personal Care & Health',

    'BABY NEEDS': 'Baby',
    'PET FOOD': 'Pet',

    'LIQUOR': 'Liquor',
    'Australian Beer': 'Liquor',
    'Premix Drinks': 'Liquor',
    'Shiraz': 'Liquor',
    'Craft Beer': 'Liquor',
    'Whisky': 'Liquor',

    'GENERAL MERCHANDISE': 'General Merchandise',
    'BASIC APPAREL': 'General Merchandise',
    'APPAREL OUTERWEAR': 'General Merchandise',
    'IN-STORE BH/BOUTIQUE': 'General Merchandise'
}

# Apply mapping
master_df['main_category'] = master_df['raw_main_cat'].map(category_map).fillna('Other')

In [54]:
print(master_df['main_category'].value_counts(dropna=False))

main_category
Other                     24659
Personal Care & Health     7666
Pantry                     6301
Snacks & Confectionery     4475
Dairy & Refrigerated       2830
Cleaning & Household       2593
Beverages                   945
Liquor                      815
General Merchandise         773
Bakery                      648
Pet                         554
Baby                        425
Fresh Food                  359
Frozen                      349
Meat & Seafood              163
Name: count, dtype: int64


In [55]:
print(master_df['main_category'].value_counts(dropna=False))

main_category
Other                     24659
Personal Care & Health     7666
Pantry                     6301
Snacks & Confectionery     4475
Dairy & Refrigerated       2830
Cleaning & Household       2593
Beverages                   945
Liquor                      815
General Merchandise         773
Bakery                      648
Pet                         554
Baby                        425
Fresh Food                  359
Frozen                      349
Meat & Seafood              163
Name: count, dtype: int64


In [56]:
print(master_df[master_df['main_category'] == 'Other']['raw_main_cat'].value_counts(dropna=False).head(30))

raw_main_cat
NaN                                16722
BREAKFAST FOODS                      224
SNACKS                               208
CANNED FISH                          189
ICE CREAM                            174
DOMESTICWARE                         166
LIFESTYLE/WATER NON CARBONATED       164
CANNED VEGETABLES                    154
INSTORE BAKERY                       133
PAPERGOODS                           126
SEASONAL FOODS                       119
HOUSEHOLD CLEANING                   114
CHEESE EVERYDAY                      110
GARDEN AIDS/SEEDS/BULBS              108
JAMS / SPREADS                       106
CANNED FRUIT / DESSERTS              102
DAIRY - MILK                          98
DAIRY - CHILLED JUICES & DRINKS       94
MEAT CONVENIENCE                      88
VEG / FRESHCUTS / HARD PRODUCE        88
LONGLIFE JUICE / DRINKS               85
APPAREL                               83
HABERDASHERY                          78
CORDIAL / DRINK BASES                 75
FRE

In [57]:
print("Top unmapped raw categories:")
print(master_df[master_df['main_category'] == 'Other']['raw_main_cat'].value_counts(dropna=False).head(30))

Top unmapped raw categories:
raw_main_cat
NaN                                16722
BREAKFAST FOODS                      224
SNACKS                               208
CANNED FISH                          189
ICE CREAM                            174
DOMESTICWARE                         166
LIFESTYLE/WATER NON CARBONATED       164
CANNED VEGETABLES                    154
INSTORE BAKERY                       133
PAPERGOODS                           126
SEASONAL FOODS                       119
HOUSEHOLD CLEANING                   114
CHEESE EVERYDAY                      110
GARDEN AIDS/SEEDS/BULBS              108
JAMS / SPREADS                       106
CANNED FRUIT / DESSERTS              102
DAIRY - MILK                          98
DAIRY - CHILLED JUICES & DRINKS       94
MEAT CONVENIENCE                      88
VEG / FRESHCUTS / HARD PRODUCE        88
LONGLIFE JUICE / DRINKS               85
APPAREL                               83
HABERDASHERY                          78
CORDIAL / DRINK

In [58]:
print("Other category by retailer:")
print(master_df[master_df['main_category'] == 'Other']['retailer'].value_counts())

Other category by retailer:
retailer
Woolworths    20406
IGA            4246
Coles             7
Name: count, dtype: int64


## 10. Category Refinement and Coverage Improvement

After the first-pass mapping, remaining unmapped categories were reviewed and refined. This helped reduce the size of the residual “Other” category and improve the overall usefulness of the category framework.

At this stage, the mapping was considered strong enough for a V1 feature, even though some retailer-specific category gaps remained.

In [60]:
# Additional category refinement for unmapped raw categories

extra_category_map = {
    'BREAKFAST FOODS': 'Pantry',
    'SNACKS': 'Snacks & Confectionery',
    'CANNED FISH': 'Pantry',
    'ICE CREAM': 'Frozen',
    'DOMESTICWARE': 'General Merchandise',
    'LIFESTYLE/WATER NON CARBONATED': 'Beverages',
    'CANNED VEGETABLES': 'Pantry',
    'INSTORE BAKERY': 'Bakery',
    'PAPERGOODS': 'Cleaning & Household',
    'SEASONAL FOODS': 'Pantry',
    'HOUSEHOLD CLEANING': 'Cleaning & Household',
    'CHEESE EVERYDAY': 'Dairy & Refrigerated',
    'GARDEN AIDS/SEEDS/BULBS': 'General Merchandise',
    'JAMS / SPREADS': 'Pantry',
    'CANNED FRUIT / DESSERTS': 'Pantry',
    'DAIRY - MILK': 'Dairy & Refrigerated',
    'DAIRY - CHILLED JUICES & DRINKS': 'Beverages',
    'MEAT CONVENIENCE': 'Meat & Seafood',
    'VEG / FRESHCUTS / HARD PRODUCE': 'Fresh Food',
    'LONGLIFE JUICE / DRINKS': 'Beverages',
    'APPAREL': 'General Merchandise',
    'HABERDASHERY': 'General Merchandise',
    'CORDIAL / DRINK BASES': 'Beverages',
    'FREEZER - FISH': 'Frozen',
    'DAIRY - ENTERTAINING': 'Dairy & Refrigerated',
    'STATIONERY': 'General Merchandise',
    'CHEESE ENTERTAINING': 'Dairy & Refrigerated',
    'Sauvignon Blanc': 'Liquor',
    'Dips and Pate': 'Dairy & Refrigerated'
}

# Update only rows currently still marked as Other
mask_other = master_df['main_category'] == 'Other'
master_df.loc[mask_other, 'main_category'] = (
    master_df.loc[mask_other, 'raw_main_cat']
    .map(extra_category_map)
    .fillna(master_df.loc[mask_other, 'main_category'])
)

In [61]:
print(master_df['main_category'].value_counts(dropna=False))

main_category
Other                     21422
Personal Care & Health     7666
Pantry                     7195
Snacks & Confectionery     4683
Dairy & Refrigerated       3211
Cleaning & Household       2833
Beverages                  1363
General Merchandise        1266
Liquor                      869
Bakery                      781
Frozen                      589
Pet                         554
Fresh Food                  447
Baby                        425
Meat & Seafood              251
Name: count, dtype: int64


In [62]:
print("Top unmapped raw categories after refinement:")
print(master_df[master_df['main_category'] == 'Other']['raw_main_cat'].value_counts(dropna=False).head(30))

Top unmapped raw categories after refinement:
raw_main_cat
NaN                                 16722
FREEZER - VEGETABLES                   53
DAIRY - BUTTER & MARGARINE             51
Chardonnay                             51
Lifestyle Beverages                    51
Tea Bags                               49
Flavoured Yoghurt                      48
FREEZER - POULTRY                      48
Pinot Noir                             48
Vodka                                  48
Muesli and Oats                        46
NEWSAGENCY                             46
Bourbon                                44
Cabernet Sauvignon                     44
Cereal Bars & Nutritional Snacks       42
Trays and Can Cat Food                 42
Other Sauces                           42
Champagne & Sparkling                  42
Ice Cream Tubs                         41
Recipe Bases and Cooking Sauces        41
OILS (COOKING)                         40
Oils and Vinegar                       39
Trays and Can Dog

In [63]:
print("Other category by retailer after refinement:")
print(master_df[master_df['main_category'] == 'Other']['retailer'].value_counts())

Other category by retailer after refinement:
retailer
Woolworths    17277
IGA            4138
Coles             7
Name: count, dtype: int64


## 11. Exporting the Harmonised Master Dataset

At this checkpoint, the harmonised master dataset is exported for manual review in Excel. This makes it easier to validate the category mapping, price fields, and derived features before moving into the scoring stage.

In [64]:
master_df.to_csv(
    r"C:\Shazza\T1 2026\Capstone Project B\#_True_Value_Promotion\Datasets_23_Mar\master_true_value_dataset_v1.csv",
    index=False
)

print("File saved successfully.")

File saved successfully.


## 12. Creating the Promotion Scoring Dataset

Not every product in the master dataset needs to be scored. In this section, a promotion-only working dataset is created and then restricted to products with measurable discounts.

This was an important decision in the feature design, because category benchmarks and scores should be based on real discount values rather than retailer labels alone.

In [66]:
# Step 1: Create promo-only dataset
promo_df = master_df[
    (master_df['is_promoted'] == 1) | (master_df['has_measurable_saving'] == 1)
].copy()

# Keep only rows with a valid current price
promo_df = promo_df[promo_df['current_price'].notna()].copy()

# Fill numeric fields safely
promo_df['saving_amount'] = promo_df['saving_amount'].fillna(0)
promo_df['discount_percent'] = promo_df['discount_percent'].fillna(0)

# Quick check
print("Promo dataset shape:", promo_df.shape)
print(promo_df[['retailer', 'product_name', 'main_category',
                'current_price', 'original_price',
                'saving_amount', 'discount_percent']].head(10))

Promo dataset shape: (12301, 22)
   retailer             product_name main_category  current_price  \
0     Coles             Strawberries    Fresh Food           4.50   
2     Coles                  Carrots    Fresh Food           1.70   
3     Coles          Iceberg Lettuce    Fresh Food           2.90   
21    Coles         Shepard Avocados    Fresh Food           1.50   
22    Coles          Broccoli Medium    Fresh Food           1.53   
33    Coles          Cherry Tomatoes    Fresh Food           2.50   
37    Coles              Raspberries    Fresh Food           4.00   
43    Coles             Orange Navel    Fresh Food           1.98   
45    Coles  Lettuce Cos Baby Hearts    Fresh Food           2.50   
49    Coles                   Lemons    Fresh Food           1.80   

    original_price  saving_amount  discount_percent  
0              NaN            0.0               0.0  
2              NaN            0.0               0.0  
3              NaN            0.0            

In [124]:
# Keep only rows with measurable discount for scoring
promo_df = promo_df[promo_df['discount_percent'] > 0].copy()

# Reset index for cleanliness
promo_df = promo_df.reset_index(drop=True)

# Quick check
print("Scoring promo dataset shape:", promo_df.shape)
print(promo_df[['retailer', 'product_name', 'main_category',
                'current_price', 'original_price',
                'saving_amount', 'discount_percent']].head(10))

Scoring promo dataset shape: (10771, 22)
  retailer                               product_name         main_category  \
0    Coles                         Regular Pork Mince        Meat & Seafood   
1    Coles   Tasmanian Salmon Portions Skin On 4 Pack        Meat & Seafood   
2    Coles                       Lamb Loin Chops Bulk        Meat & Seafood   
3    Coles                            Pork Spare Ribs        Meat & Seafood   
4    Coles  RSPCA Approved Chicken Kebabs Honey & Soy            Fresh Food   
5    Coles                        Beef Sandwich Steak        Meat & Seafood   
6    Coles                      Beef & Pork Meatballs            Fresh Food   
7    Coles                            Short Cut Bacon  Dairy & Refrigerated   
8    Coles                          Beef Sizzle Steak        Meat & Seafood   
9    Coles       RSPCA Approved Chicken Thigh Cutlets        Meat & Seafood   

   current_price  original_price  saving_amount  discount_percent  
0           5.50     

## 13. Calculating Category-Level Discount Benchmarks

To make the feature category-aware, discount behaviour is analysed within each main category. Metrics such as average discount, median discount, and quartiles are calculated to provide category context.

This supports a more realistic interpretation of value, since a strong discount in Fresh Food may look different from a strong discount in Cleaning or Personal Care.

In [128]:
# Step 2: Category-level discount benchmarks
category_stats = promo_df.groupby('main_category')['discount_percent'].agg(
    category_avg_discount='mean',
    category_median_discount='median',
    category_max_discount='max'
).reset_index()

# Add quartiles
category_quantiles = promo_df.groupby('main_category')['discount_percent'].quantile([0.25, 0.75]).unstack().reset_index()
category_quantiles.columns = ['main_category', 'category_q25_discount', 'category_q75_discount']

# Merge all category benchmark stats
category_stats = category_stats.merge(category_quantiles, on='main_category', how='left')

print(category_stats)

             main_category  category_avg_discount  category_median_discount  \
0                     Baby              18.202857                    13.040   
1                   Bakery              18.951667                    18.065   
2                Beverages              28.966295                    30.000   
3     Cleaning & Household              35.160585                    33.330   
4     Dairy & Refrigerated              22.415592                    20.230   
5               Fresh Food              18.130000                    15.480   
6                   Frozen              30.391871                    30.000   
7      General Merchandise              38.963473                    50.000   
8                   Liquor              14.499219                    13.300   
9           Meat & Seafood              16.378889                    13.640   
10                   Other              18.745192                    14.290   
11                  Pantry              26.714037   

In [134]:
# Step 3: Merge category benchmark stats into promo_df
promo_df = promo_df.merge(category_stats, on='main_category', how='left')

# Quick check
print(promo_df[['product_name', 'main_category', 'discount_percent',
                'category_avg_discount', 'category_median_discount',
                'category_q25_discount', 'category_q75_discount']].head(10))

                                product_name         main_category  \
0                         Regular Pork Mince        Meat & Seafood   
1   Tasmanian Salmon Portions Skin On 4 Pack        Meat & Seafood   
2                       Lamb Loin Chops Bulk        Meat & Seafood   
3                            Pork Spare Ribs        Meat & Seafood   
4  RSPCA Approved Chicken Kebabs Honey & Soy            Fresh Food   
5                        Beef Sandwich Steak        Meat & Seafood   
6                      Beef & Pork Meatballs            Fresh Food   
7                            Short Cut Bacon  Dairy & Refrigerated   
8                          Beef Sizzle Steak        Meat & Seafood   
9       RSPCA Approved Chicken Thigh Cutlets        Meat & Seafood   

   discount_percent  category_avg_discount  category_median_discount  \
0             15.38              16.378889                     13.64   
1             10.26              16.378889                     13.64   
2            

## 14. Creating the Base Promotion Score

A base score is created using both discount percentage and saving amount. During development, this logic was revised so that very large dollar savings on high-priced products would not dominate the ranking unfairly.

For the final V1 version, saving amount is capped before scoring, which keeps absolute savings relevant while giving stronger emphasis to discount percentage.

In [142]:
# Revised Step 4: Base promotion score with capped saving amount
promo_df['saving_amount_capped'] = promo_df['saving_amount'].clip(upper=20)

promo_df['base_score'] = (
    (promo_df['saving_amount_capped'] * 0.2) +
    (promo_df['discount_percent'] * 0.8)
)

promo_df['base_score'] = promo_df['base_score'].round(2)

print(promo_df[['product_name', 'saving_amount', 'saving_amount_capped',
                'discount_percent', 'base_score']].head(10))

                                product_name  saving_amount  \
0                         Regular Pork Mince           1.00   
1   Tasmanian Salmon Portions Skin On 4 Pack           2.00   
2                       Lamb Loin Chops Bulk           6.23   
3                            Pork Spare Ribs           2.85   
4  RSPCA Approved Chicken Kebabs Honey & Soy           1.50   
5                        Beef Sandwich Steak           2.00   
6                      Beef & Pork Meatballs           1.00   
7                            Short Cut Bacon           0.70   
8                          Beef Sizzle Steak           2.00   
9       RSPCA Approved Chicken Thigh Cutlets           1.10   

   saving_amount_capped  discount_percent  base_score  
0                  1.00             15.38       12.50  
1                  2.00             10.26        8.61  
2                  6.23             21.21       18.21  
3                  2.85             13.64       11.48  
4                  1.50   

## 15. Adding a Category-Relative Score

A category-relative score is added to reflect how strong a product’s discount is compared with other discounts in the same category. Products above the upper quartile receive a higher relative score than products closer to the category median or lower quartile.

This step makes the feature more intelligent by combining direct promotion strength with category-normalised discount behaviour.

In [138]:
# Step 5: Category-relative score
def category_relative_score(row):
    dp = row['discount_percent']
    q25 = row['category_q25_discount']
    median = row['category_median_discount']
    q75 = row['category_q75_discount']
    
    if pd.isna(dp):
        return 0
    elif dp >= q75:
        return 15
    elif dp >= median:
        return 10
    elif dp >= q25:
        return 5
    else:
        return 0

promo_df['category_relative_score'] = promo_df.apply(category_relative_score, axis=1)

print(promo_df[['product_name', 'main_category', 'discount_percent',
                'category_q25_discount', 'category_median_discount',
                'category_q75_discount', 'category_relative_score']].head(10))

                                product_name         main_category  \
0                         Regular Pork Mince        Meat & Seafood   
1   Tasmanian Salmon Portions Skin On 4 Pack        Meat & Seafood   
2                       Lamb Loin Chops Bulk        Meat & Seafood   
3                            Pork Spare Ribs        Meat & Seafood   
4  RSPCA Approved Chicken Kebabs Honey & Soy            Fresh Food   
5                        Beef Sandwich Steak        Meat & Seafood   
6                      Beef & Pork Meatballs            Fresh Food   
7                            Short Cut Bacon  Dairy & Refrigerated   
8                          Beef Sizzle Steak        Meat & Seafood   
9       RSPCA Approved Chicken Thigh Cutlets        Meat & Seafood   

   discount_percent  category_q25_discount  category_median_discount  \
0             15.38                10.3950                     13.64   
1             10.26                10.3950                     13.64   
2            

In [144]:
# Recalculate final score after revised base score
promo_df['true_value_score'] = promo_df['base_score'] + promo_df['category_relative_score']
promo_df['true_value_score'] = promo_df['true_value_score'].round(2)

# Recalculate discount class
def classify_discount(score):
    if score >= 45:
        return 'Excellent Discount'
    elif score >= 25:
        return 'Good Discount'
    elif score > 0:
        return 'OK Discount'
    else:
        return 'Low Impact'

promo_df['discount_class'] = promo_df['true_value_score'].apply(classify_discount)

# Rebuild sorted deals table
top_deals = promo_df.sort_values(by='true_value_score', ascending=False).copy()

# Print fresh results
print(promo_df['discount_class'].value_counts())

print(top_deals[[
    'retailer', 'product_name', 'main_category',
    'current_price', 'original_price',
    'saving_amount', 'saving_amount_capped',
    'discount_percent', 'base_score',
    'category_relative_score', 'true_value_score',
    'discount_class'
]].head(20))

discount_class
OK Discount           4089
Good Discount         3746
Excellent Discount    2936
Name: count, dtype: int64
        retailer                                       product_name  \
1558       Coles                      Alkalising Greens Berry Burst   
1560       Coles                     Alkalising Greens Citrus Twist   
9043  Woolworths      Tango Of The Bulldogs String Orchestra Gr 2.5   
8168  Woolworths                   Music Of Smash Concert Band Gr 3   
6211  Woolworths         Campbells Are Coming String Orchestra Gr 2   
5531  Woolworths  Circulon Steelshield C Series 36cm Covered Wok...   
1100       Coles                           Kids Toothbrush Ages 5-7   
3841       Coles      Dog Treats Brekkie Bacon & Egg Flavour Strapz   
3626       Coles                     Australian Sun Muscat Sultanas   
245        Coles                360 Advanced Soft Manual Toothbrush   
5828  Woolworths  Oral-B Pro 100 Electric Toothbrush Extra Sensi...   
8916  Woolworths          

In [146]:
print(
    promo_df[['product_name', 'saving_amount', 'saving_amount_capped',
              'discount_percent', 'base_score',
              'category_relative_score', 'true_value_score']]
    .sort_values(by='true_value_score', ascending=False)
    .head(10)
)

                                           product_name  saving_amount  \
1558                      Alkalising Greens Berry Burst          36.00   
1560                     Alkalising Greens Citrus Twist          36.00   
9043      Tango Of The Bulldogs String Orchestra Gr 2.5          88.45   
8168                   Music Of Smash Concert Band Gr 3         147.95   
6211         Campbells Are Coming String Orchestra Gr 2          85.95   
5531  Circulon Steelshield C Series 36cm Covered Wok...         290.00   
1100                           Kids Toothbrush Ages 5-7           2.55   
3841      Dog Treats Brekkie Bacon & Egg Flavour Strapz           5.50   
3626                     Australian Sun Muscat Sultanas           2.70   
245                 360 Advanced Soft Manual Toothbrush           4.00   

      saving_amount_capped  discount_percent  base_score  \
1558                 20.00             80.00       68.00   
1560                 20.00             80.00       68.00   
9043 

In [148]:
# Create a customer-facing grocery-focused deals dataset
customer_deals_df = promo_df[promo_df['main_category'] != 'Other'].copy()

print(customer_deals_df['discount_class'].value_counts())

top_customer_deals = customer_deals_df.sort_values(by='true_value_score', ascending=False)

print(top_customer_deals[[
    'retailer', 'product_name', 'main_category',
    'current_price', 'original_price',
    'saving_amount', 'saving_amount_capped',
    'discount_percent', 'base_score',
    'category_relative_score', 'true_value_score',
    'discount_class'
]].head(20))

discount_class
Good Discount         3182
OK Discount           3050
Excellent Discount    2765
Name: count, dtype: int64
        retailer                                       product_name  \
1560       Coles                     Alkalising Greens Citrus Twist   
1558       Coles                      Alkalising Greens Berry Burst   
1100       Coles                           Kids Toothbrush Ages 5-7   
3841       Coles      Dog Treats Brekkie Bacon & Egg Flavour Strapz   
3626       Coles                     Australian Sun Muscat Sultanas   
245        Coles                360 Advanced Soft Manual Toothbrush   
5828  Woolworths  Oral-B Pro 100 Electric Toothbrush Extra Sensi...   
2775       Coles                           Sour Patch Kids Biscuits   
1109       Coles  Pro 1500x Sensitive Clean Electric Toothbrush ...   
2370       Coles                             Car Charger Dual Usb-a   
1110       Coles  3d White Diamond Clean Clinical Whitening Toot...   
1107       Coles  3d White

## 16. Final True Value Score and Customer-Facing Labels

The final True Value Score combines the base promotion score with the category-relative score. This score is then translated into customer-facing labels such as OK Discount, Good Discount, Excellent Discount, and DiscountMate Recommends.

A grocery-focused view is also created by excluding products in the residual “Other” category, producing a cleaner customer-facing output for the feature.

At this stage, the feature moved from internal scoring logic to a more user-facing format. The goal was not only to rank deals mathematically, but also to express the result in a way that could be clearly displayed to customers.

In [153]:
# Create a more selective customer-facing label
def classify_discount_v2(score):
    if score >= 70:
        return 'DiscountMate Recommends'
    elif score >= 45:
        return 'Excellent Discount'
    elif score >= 25:
        return 'Good Discount'
    elif score > 0:
        return 'OK Discount'
    else:
        return 'Low Impact'

customer_deals_df['discount_class_v2'] = customer_deals_df['true_value_score'].apply(classify_discount_v2)

print(customer_deals_df['discount_class_v2'].value_counts())

top_customer_deals_v2 = customer_deals_df.sort_values(by='true_value_score', ascending=False)

print(top_customer_deals_v2[[
    'retailer', 'product_name', 'main_category',
    'current_price', 'original_price',
    'saving_amount', 'discount_percent',
    'true_value_score', 'discount_class_v2'
]].head(20))

discount_class_v2
Good Discount              3182
OK Discount                3050
Excellent Discount         2761
DiscountMate Recommends       4
Name: count, dtype: int64
        retailer                                       product_name  \
1560       Coles                     Alkalising Greens Citrus Twist   
1558       Coles                      Alkalising Greens Berry Burst   
1100       Coles                           Kids Toothbrush Ages 5-7   
3841       Coles      Dog Treats Brekkie Bacon & Egg Flavour Strapz   
3626       Coles                     Australian Sun Muscat Sultanas   
245        Coles                360 Advanced Soft Manual Toothbrush   
5828  Woolworths  Oral-B Pro 100 Electric Toothbrush Extra Sensi...   
2775       Coles                           Sour Patch Kids Biscuits   
1109       Coles  Pro 1500x Sensitive Clean Electric Toothbrush ...   
2370       Coles                             Car Charger Dual Usb-a   
1110       Coles  3d White Diamond Clean Clinic

## 17. Exporting Final Feature Outputs

#### a)Export the final customer-facing dataset

In [157]:
customer_deals_df.to_csv(
    r"C:\Shazza\T1 2026\Capstone Project B\#_True_Value_Promotion\Datasets_23_Mar\true_value_customer_deals_v1.csv",
    index=False
)

print("Customer-facing deals dataset saved successfully.")

Customer-facing deals dataset saved successfully.


#### b)Save the full scored promo dataset too

In [159]:
promo_df.to_csv(
    r"C:\Shazza\T1 2026\Capstone Project B\#_True_Value_Promotion\Datasets_23_Mar\true_value_scored_full_v1.csv",
    index=False
)

print("Full scored promo dataset saved successfully.")

Full scored promo dataset saved successfully.


The final V1 outputs are exported for review and downstream use. These include the full scored promotion dataset and the grocery-focused customer-facing deals dataset.

These files represent the completed first version of the True Value Promotion feature.